# Study notebook: `augmentation.py`

This notebook contains the exact production code from the matching Python module. Each explanation appears immediately before the code section it explains. The code is copied line-for-line from the application module.

## `imports and module setup`

This section is part of the production implementation and supports the module's main workflow.

In [ ]:
"""Turn retrieved records into a grounded, citation-friendly prompt."""


## `imports and module setup`

This section is part of the production implementation and supports the module's main workflow.

In [ ]:
from __future__ import annotations



## `imports and module setup`

This section is part of the production implementation and supports the module's main workflow.

In [ ]:
import json




## `build_context`

Converts retrieved records into bounded evidence blocks, stable source tags, citations, and confidence.

In [ ]:
def build_context(question: str, records: list[dict], max_chars: int = 9000) -> dict:
    # Turn search results into short evidence blocks with stable source tags.
    blocks: list[str] = []
    citations: list[dict] = []
    used = 0
    for index, record in enumerate(records, start=1):
        source_tag = f"S{index}"
        text = record.get("text", "").strip()
        block = f"[{source_tag}] {text}"
        if used + len(block) > max_chars:
            break
        blocks.append(block)
        citations.append({
            "tag": source_tag,
            "source": record.get("source", "Unknown"),
            "url": record.get("url", ""),
            "id": record.get("id", ""),
            "page": record.get("metadata", {}).get("page"),
            "local_file": record.get("metadata", {}).get("local_file", ""),
            "similarity": record.get("similarity", 0.0),
            "reference_only": record.get("reference_only", False),
        })
        used += len(block)
    # These rules help the language model stay close to the retrieved evidence.
    instructions = (
        "Answer the question using the evidence below. Cite factual claims with [S#]. "
        "If the evidence is insufficient, say so clearly; do not invent a Gardiner value, "
        "translation, date, or historical claim. Preserve uncertainty and distinguish a sign's "
        "shape, phonetic value, word meaning, and cultural interpretation. "
        "Answer in the user's language, in a natural helpful tone. If no records are supplied, "
        "say that the local knowledge base has no verified evidence instead of guessing."
    )
    confidence = max((float(r.get("similarity", 0)) for r in records), default=0.0)
    return {"question": question, "instructions": instructions, "context": "\n\n".join(blocks), "citations": citations,
            "retrieval_note": "Evidence found; verify interpretation against cited sources." if records else "No matching evidence found.",
            "confidence": round(confidence, 3)}




## `prompt_messages`

Creates the provider-ready system and user messages from the evidence packet.

In [ ]:
def prompt_messages(question: str, records: list[dict]) -> list[dict]:
    packet = build_context(question, records)
    return [
        {"role": "system", "content": packet["instructions"]},
        {"role": "user", "content": json.dumps(packet, ensure_ascii=False)},
    ]

